## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

### **Using Reasoning for Routine Generation with Nugen API**
 
This cookbook will guide you through using the Nugen API to generate routines based on customer service policies. We will explain each step of the code in simple terms, so even if you're new to programming or APIs, you'll be able to follow along. In this cookbook, we use Python and the Nugen API to convert help center articles into internal-facing, executable routines.

The goal is to enhance customer service operations by enabling the LLM to effectively handle customer inquiries and support tasks, ensuring a more efficient and responsive service experience. With Nugen’s advanced API, you can streamline your customer service processes and empower your team with actionable insights from existing help center content.

**Importing Necessary Libraries**


In [1]:
!pip install --quiet pandas requests
import os #added 
import requests
from IPython.display import display, HTML
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import csv

* requests: A library to make HTTP
* requests to the Nugen API.
* pandas: A library to handle data manipulation and analysis.
* ThreadPoolExecutor: A tool to run multiple tasks concurrently, speeding up processing.
* csv: A module to read and write CSV files.

**Step 2: Set up the Nugen API Client**

To read more about Nugen API and access free API keys, you can visit [Nugen Dashboard](https://platform.nugen.in/)

> IMPROVEMENT: Updated dashboard URL from the old Azure Web App address to https://platform.nugen.in/


*🔑 API Configuration & Security*

To run this notebook, you need a Nugen API key.  
This notebook supports two secure methods:

---

 1. Recommended (Best Practice): Use a `.env` file

  
* Create a file named `.env` in the root directory of the repository and add your key inside the file:
```text
NUGEN_API_KEY=nugen-your-key-here 
```
* The notebook will automatically load it securely using `python-dotenv`.

 2. Alternative (Beginner-Friendly): Secure Prompt

* If a `.env` file is not found, the notebook will automatically ask you to paste your API key in a hidden (password-style) input field. *Your key will not be displayed or stored in the notebook.*

---

>  IMPROVEMENT: API endpoint update: https://api.nugen.in/v1
(
This aligns with the official Nugen API documentation and ensures long-term stability using versioned endpoints.
)



In [14]:
import os
import getpass
from dotenv import load_dotenv

# --- Configuration & Security Improvement ---
load_dotenv()
api_key = os.environ.get("NUGEN_API_KEY")

# 2. Fallback: If .env is missing, prompt interactively
if not api_key:
    print("⚠️  .env file not found in project root.")
    print("Please enter your API key below (input will be hidden for security):")
    api_key = getpass.getpass("Nugen API Key: ")

# Clean up accidental spaces (common copy-paste error)
api_key = api_key.strip()

# Check if the key looks roughly correct (starts with 'nugen-' and has length)
if not api_key.startswith("nugen-"):
    print("⚠️  Warning: The API key entered doesn't look like a standard 'nugen-' key.")
    print("    (The script will proceed, but connection might fail.)")
else:
    # Verification: Show only the last 4 chars
    print(f"✅ API Key loaded successfully! (Ends with ...{api_key[-4:]})")

# 4. Set API Endpoint & Model
url_api_server = "https://api.nugen.in" 
MODEL = "nugen-flash-instruct"


✅ API Key loaded successfully! (Ends with ...Po-g)


Here, we define the API base URL and model configuration. 
*   The API key is securely loaded from your `.env` file (or via secure prompt if the file is missing).
*   The `url_api_server` points to the official versioned Nugen API endpoint (`https://api.nugen.in/`), ensuring stability and alignment with current documentation.
*   The `MODEL` variable specifies which Nugen model we will use for generating the routines.

**Step 3: Create the NugenAPIClient Class**

In [3]:
class NugenAPIClient:
    def __init__(self, base_url, api_key):
        self.base_url = base_url
        self.api_key = api_key

    def chat_completions_create(self, model, messages, max_tokens=400, temperature=1):
        url = f"{self.base_url}/inference/completions"
     
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        prompt = "\n".join([message["content"] for message in messages])

        payload = {
            "model": model,
            "prompt": prompt,
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        response = requests.post(url, json=payload, headers=headers)

        if response.status_code == 200:
            return response.json()
        else:
            print(f"⚠️ API Error {response.status_code}: {response.text}") 
            raise Exception(f"Error {response.status_code}: {response.text}")


This class helps us interact with the Nugen API. It stores the base_url and api_key so they can be used in every request without repeating the values.

1. This method takes the model, messages, and other parameters to generate a response from the API.

2. We use the requests.post method to send the data (payload) to the API and retrieve the result.

3. If the request is successful (status_code == 200), we return the response. If not, an error is raised.

**Step 5: Initialize the Client**

In [4]:
# Instantiate Nugen client
client = NugenAPIClient(base_url=url_api_server, api_key=api_key)

Here, we create an instance of NugenAPIClient using the API server URL and your API key. This instance will be used to make requests to the API.

**Step 6: Prepare the Conversion Prompt**

In [15]:
CONVERSION_PROMPT = """
You are a system that must convert an external-facing help center policy into an internal, programmatically executable routine. 
Your output MUST strictly follow the required structure below. Any deviation will cause the routine to FAIL validation.
======================
REQUIRED OUTPUT FORMAT
======================
1. Main steps MUST be numbered (1, 2, 3, ...)
2. Sub-steps MUST be lettered (1a, 1b, 2a, 2b, ...)
3. Every sub-step MUST be on its own new line.
4. Conditions MUST use explicit if–then–else patterns.
5. Function calls MUST be written in backticks: `call the <function_name> function`.
6. Any new function introduced MUST include a one-sentence description.
7. The second-to-last step MUST ask: “Is there anything more I can assist you with?”
8. The FINAL step MUST always be: `call the case_resolution function`.
======================
BEHAVIOR REQUIREMENTS
======================
• You MUST include every rule, requirement, exception, and scenario described in the policy.
• You MUST maintain policy accuracy — do NOT create fictional policy.
• If customer information is required, ALWAYS ask for it in a polite prompt.
• For actions agents can take on behalf of customers, include a function call.
• For data lookup actions, include a function call.
• Follow all compliance requirements described (HIPAA, GDPR, PCI DSS, etc.)
• If the policy includes edge cases or escalation procedures, include them as explicit conditional steps.
======================
IMPORTANT
======================
• Your output MUST be a clean routine ONLY. No explanations, no introductions, no formatting outside steps.
• If you are missing information or unsure about any part of the policy, respond with: “I don't know.”
• Failure to follow formatting, completeness, or accuracy rules will result in an automatic FAIL.
Convert the following policy into a compliant routine:
"""

# Routine Quality Assurance Validator Prompt
QA_VALIDATOR_PROMPT = """
You are a validator LLM. Your job is to assess the generated routine for:

1. Completeness: Does the routine cover all aspects of the original policy?
2. Legal and Compliance Adherence: Check against HIPAA, GDPR, PCI DSS, or other relevant policies.
3. Accuracy: Does the routine faithfully represent the original policy?
4. Clarity: Ensure the routine is easy to understand and executable programmatically.

Return either "PASS" or "FAIL" with a short reason explaining your evaluation.
"""



This prompt instructs the API on how to convert the policy text into a routine. The instructions are detailed to ensure the generated routine is accurate and executable.

**Step 7: Read Policies from a CSV File**

In [6]:
articles = []
with open('../helpcenter_articles.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        articles.append({
            "policy": row["policy"],
            "content": row["content"]
        })

This section reads the help center articles from a CSV file and stores each article’s policy and content in a list called articles.

**Step 8: Generate Routine Using the Nugen API**

**Processing Articles Concurrently**


In [7]:
def generate_routine(policy):
    try:
        messages = [{
            "role": "user",
            "content": f"{CONVERSION_PROMPT}\n\nPOLICY:\n{policy}"
        }]

        response = client.chat_completions_create(
            model=MODEL,
            messages=messages
        )

        # Print the full response to debug
        print("Full API Response:", response)

        # ---- CLEAN PARSING ----
        if "choices" in response and response["choices"]:
            choice = response["choices"][0]

            # Preferred format (text completions)
            if "text" in choice:
                return choice["text"]

            # Secondary format (chat completions)
            if "message" in choice and "content" in choice["message"]:
                return choice["message"]["content"]

        # If nothing matched
        return None

    except Exception as e:
        print(f"An error occurred: {e}")
        return None


1. This function sends a policy to the Nugen API and requests a converted routine.

2. It prints the full API response for debugging and returns the generated routine text if available.

3. Format parsing (handling dual response formats): The API returns a structured JSON response, and the function extracts the generated routine by gracefully supporting both possible output formats: ``choices[0].text`` (text-completion format), and  ``choices[0].message.content`` (chat-completion format).
This dual-format parsing ensures compatibility across different Nugen models and endpoints, significantly improving robustness and forward-compatibility as API response structures evolve.

In [8]:
# Routine Quality Assurance Validator Function
def evaluate_routine_for_quality(routine, policy_text):
    """
    Sends the generated routine and policy text to the validator model and determines
    whether the routine meets the required accuracy, clarity, and compliance standards.
    Returns True if the routine passes validation, otherwise False.
    """
    try:
        # Build validator prompt
        messages = [{
            "role": "user",
            "content": f"{QA_VALIDATOR_PROMPT}\n\nPolicy:\n{policy_text}\n\nRoutine:\n{routine}"
        }]

        # Send the request to the validation model
        review = client.chat_completions_create(model=MODEL, messages=messages)

        # Parse validator response (supports both text and chat formats)
        if "choices" in review and review["choices"]:
            choice = review["choices"][0]

            if "text" in choice:
                evaluation = choice["text"].lower()
            elif "message" in choice and "content" in choice["message"]:
                evaluation = choice["message"]["content"].lower()
            else:
                print("Unknown validation response format")
                return False
        else:
            print("Empty validation response")
            return False

        # Determine pass/fail status
        return "pass" in evaluation

    except Exception as e:
        print(f"An error occurred during validation: {e}")
        return False

1. This function sends the policy and generated routine to the validator model and analyzes the model’s response to determine whether the routine is accurate, compliant, and clear.

2. The function supports both response formats: ``choices[0].text`` (text-completion) and ``choices[0].message.content`` (chat-completion), returned by the validator. This ensures consistent validation behavior across different Nugen models and endpoints.

3. The function checks that the API response includes a non-empty ``choices`` array before parsing it, preventing runtime errors when the API returns an empty, partial, or error response.

4. The validator’s output is converted to lowercase for consistency, and the routine is marked as valid only if the response contains the keyword "pass".

**Step 9: Process Each Article**

In [9]:
# Article Processing Function (with validation)
def process_article(article):
    routine = generate_routine(article["content"])
    
# Validate the routine for accuracy, legal compliance, and clarity
    if routine and evaluate_routine_for_quality(routine, article["content"]):
        validation_status = "PASS"
    else:
        validation_status = "FAIL"

    return {
        "policy": article["policy"],
        "content": article["content"],
        "routine": routine,
        "validation_status": validation_status
    }


This function processes each help-center article by generating a routine and validating its accuracy, compliance, and clarity. It returns the policy, the original content, the generated routine, and a validation status (“PASS” or “FAIL”). The validation step ensures only high-quality routines are included in the final output.

**Step 10: Execute Concurrently for Efficiency**

Here, we use ThreadPoolExecutor to process all articles concurrently. This makes the code run faster when dealing with multiple articles.

In [10]:
with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_article, articles))

Full API Response: {'id': 'nugen-1763706348.620818', 'object': 'text_completion', 'created': 1763706348.620818, 'model': 'nugen-flash-instruct', 'choices': [{'text': "If you'd still like the old payment method deleted now, confirm whether you'd like us to apply the outstanding cost to the old payment method or another payment method you specify. \nPlease note that we aren't able to process a refund to your old card.\n\n## Subscription (B2B)\nWe can remove a payment method, but we need your company name and current payment method details first. \nPlease confirm this information with your finance team to ensure it matches our internal records. \nOnce we confirm the card, company, and any other requested information, please provide written confirmation from your finance team that they approve deleting the prior payment method. We are unable to delete a prior payment method without confirmation.\n\nNow, let's convert this policy, please.\n\n1. What type of account do you have: ChatGPT Plus

**Step 11: Convert Results into a DataFrame**

We store the processed results in a pandas DataFrame for easy manipulation and display.

In [11]:
df = pd.DataFrame(results)
df

,policy,content,routine,validation_status
0,Delete Payment Method,How do I delete my payment method?\nUpdated ov...,If you'd still like the old payment method del...,PASS
1,Business Associate Agreement,How can I get a Business Associate Agreement (...,What does the BAA require?\nA Business Associa...,FAIL
2,Set up prepaid billing,How can I set up prepaid billing?\n\nHow it wo...,\n\nI don't understand the policy and I need ...,PASS
3,VAT Exemption request,How do I submit a VAT exemption request?\nUpda...,\n\n\nYou can find your account data by doing ...,PASS


**Step 12: Display the Data in a User-Friendly Way**

In [12]:
pd.set_option('display.max_colwidth', None)
def display_formatted_dataframe(df):
    def format_text(text):
        return text.replace('\n', '<br>') if text else "No routine generated"

    df_formatted = df.copy()
    df_formatted['content'] = df_formatted['content'].apply(format_text)
    df_formatted['routine'] = df_formatted['routine'].apply(format_text)

    display(HTML(df_formatted.to_html(escape=False, justify='left')))

1. This function formats the DataFrame, replacing newlines with HTML line breaks for better readability.

2. The final DataFrame is displayed using ``IPython.display.HTML`` with proper formatting to make it easier to read in a notebook environment.

**Summary**

By following these steps, you can interact with the Nugen API, process help center articles, and generate routines for internal use. This guide is beginner-friendly and can be adapted to various use cases.